In [5]:
from datetime import date, timedelta

import netCDF4 as nc
import numpy as np
import pandas as pd
from tqdm import tqdm

FILE_PATH = "OFAM3/temp_ofam3_7d_197901-201412.0p25x0p25.nc"

In [3]:
def load_data():
    """ Load the data from the NetCDF file
    Args:
        None

    Returns:
        temps (array): array with shape (len(dates), 1, len(lats), len(longs)) containing all the SST measurements
        dates (array): array of length 1879 containing corresponding Datetimes for the data
        lats (array): array of length 600 containing the range of latitude values for the data
        longs (array): array of length 1440 containing the range of longitude values for the data
    """
    file_data = nc.Dataset(FILE_PATH,'r')

    print(file_data["Time"][:])

    # Extract variables
    start_time = date(1979,1,1)
    dates = [start_time + timedelta(t) for t in file_data["Time"][:].data]  # Time stored as days since 1979-01-01 00:00:00
    lats = file_data["yt_ocean"][:].data  # in degrees N
    longs = file_data["xt_ocean"][:].data  # in degrees E
    # depth = file_data["st_ocean"][:].data  # in metres, Z axis
    temps = file_data["temp"][:].data  # in degrees C, sea_water_potential_temperature
    avg_time = file_data["average_DT"][:].data
    # temp.shape is (Time, st_ocean, yt_ocean, xt_ocean)

    file_data.close()

    return temps, dates, lats, longs, avg_time

In [4]:
temps, dates, lats, longs, avg_time = load_data()

[3.50000e+00 1.05000e+01 1.75000e+01 ... 1.31355e+04 1.31425e+04
 1.31475e+04]


### What date system is used?
Trying to understand what each set of SST reflects?

In [6]:
np.unique(avg_time)

array([1.])

In [9]:
print(len(avg_time))
print(len(dates))

1879
1879


In [10]:
dates[-1]

datetime.date(2014, 12, 30)

In [22]:
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [23]:
df = pd.DataFrame()
df['Date'] = pd.to_datetime(dates)
df['Weekday'] = [day_names[d.weekday()] for d in df['Date']]
df

,Date,Weekday
0,1979-01-04,Thursday
1,1979-01-11,Thursday
2,1979-01-18,Thursday
3,1979-01-25,Thursday
4,1979-02-01,Thursday
...,...,...
1874,2014-12-04,Thursday
1875,2014-12-11,Thursday
1876,2014-12-18,Thursday
1877,2014-12-25,Thursday


In [32]:
df.groupby('Weekday').size()

Weekday
Thursday    1878
Tuesday        1
dtype: int64